# 8.5.1 GPT-2 Notebook

In this section, we investigate whether an early decoder-only language model (GPT-2) can be
used for machine translation through prompting alone.
GPT-2 is trained purely as a language model and has not been adapted for instruction following.
This makes it a useful baseline for understanding how decoder-only models behave when applied
to translation without additional training
We create a small subset of the development data to quickly test different prompting strategies.
## Installation

We install the necessary libraries and create a 10 sentence version of `dev.en`

In [1]:
!pip install -q transformers accelerate sentencepiece
!wget https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/code/gpt2_translate.py

--2026-03-27 08:54:26--  https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/code/gpt2_translate.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4175 (4.1K) [text/plain]
Saving to: ‘gpt2_translate.py’

gpt2_translate.py   100%[===================>]   4.08K  --.-KB/s    in 0s      

2026-03-27 08:54:26 (57.4 MB/s) - ‘gpt2_translate.py’ saved [4175/4175]



We create a small subset of the development data to quickly test different prompting strategies.

In [2]:
!head -n 10 /kaggle/input/datasets/vincentvandeghinste/tatoeba-en-nl/dev.en > dev10.en

# Step 1. Zero-shot baseline 

We begin with a minimal instruction prompt. 

In [3]:
%run gpt2_translate.py \
  --input dev10.en \
  --output dev10.gpt2.zero.nl \
  --source-lang English \
  --target-lang Dutch \
  --model gpt2 \
  --prompt-template "Translate to Dutch: {text}\nDutch:" \
  --temperature 0 \
  --batch-size 4 \
  --show-translations \
  --show-n 5

[INFO] Loading model: gpt2


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...


---
SRC: Translate to Dutch: Nobody reads about my country .\nDutch:
HYP: I'm a Dutchman.\nDutch: I'm a Dutchman.\nDutch: I'm a Dutchman.\nDutch: I'm a Dutchman.\nDutch: I'm a Dutchman.\nDutch:

---
SRC: Translate to Dutch: I sleep in my car .\nDutch:
HYP: I sleep in my car .\nEnglish: I sleep in my car .\nFrench: I sleep in my car .\nGerman: I sleep in my car .\nGreek: I sleep in my car .\nHind

---
SRC: Translate to Dutch: There 're clean sheets under the bed .\nDutch:
HYP: \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n

---
SRC: Translate to Dutch: I have a donkey .\nDutch:
HYP: I have a donkey .\nDutch: I have a donkey .\nDutch: I have a donkey .\nDutch: I have a donkey .\nDutch: I have a donkey .\nDutch: I have a donkey .


 67%|██████▋   | 2/3 [00:02<00:00,  1.08it/s]


---
SRC: Translate to Dutch: Betty drives fast .\nDutch:
HYP: Betty drives fast .\nDutch: Betty drives fast .\nDutch: Betty drives fast .\nDutch: Betty drives fast .\nDutch: Betty drives fast .\nDutch: Betty drives fast .\nDutch: Betty drives


100%|██████████| 3/3 [00:02<00:00,  1.18it/s]

[INFO] Done. Saved to dev10.gpt2.zero.nl


Typical observations include:
* the model repeats parts of the prompt (e.g. Dutch:)
* it copies the source sentence instead of translating it
* it generates unrelated or multilingual content
* it may enter repetition loops

These behaviours show that GPT-2 is not interpreting the prompt as a task instruction, but as
text to be continued.

# Step 2. Format prompting

We now make the structure of the task more explicit.

In [4]:
%run gpt2_translate.py \
  --input dev10.en \
  --output dev10.gpt2.format.nl \
  --source-lang English \
  --target-lang Dutch \
  --model gpt2 \
  --prompt-template "English: {text}\nDutch:" \
  --temperature 0 \
  --batch-size 4 \
  --strip-after-first-line \
  --show-translations \
  --show-n 5

[INFO] Loading model: gpt2


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Translating...


 33%|███▎      | 1/3 [00:00<00:01,  1.79it/s]


---
SRC: English: Nobody reads about my country .\nDutch:
HYP: I'm a Dutchman.\nEnglish: I'm a Dutchman.\nEnglish: I'm a Dutchman.\nEnglish: I'm a Dutchman.\nEnglish: I'm a Dutchman.\nEnglish:

---
SRC: English: I sleep in my car .\nDutch:
HYP: I sleep in my car .\nEnglish: I sleep in my car .\nEnglish: I sleep in my car .\nEnglish: I sleep in my car .\nEnglish: I sleep in my car .\nEnglish:

---
SRC: English: There 're clean sheets under the bed .\nDutch:
HYP: There 're clean sheets under the bed .\nEnglish: There 're clean sheets under the bed .\nEnglish: There 're clean sheets under the bed .\nEnglish: There 're clean sheets under the bed .\n

---
SRC: English: I have a donkey .\nDutch:
HYP: I have a donkey .\nEnglish: I have a donkey .\nEnglish: I have a donkey .\nEnglish: I have a donkey .\nEnglish: I have a donkey .\nEnglish: I have a donkey .


 67%|██████▋   | 2/3 [00:01<00:00,  1.80it/s]


---
SRC: English: Betty drives fast .\nDutch:
HYP: Betty drives fast .\nEnglish: Betty drives fast .\nEnglish: Betty drives fast .\nEnglish: Betty drives fast .\nEnglish: Betty drives fast .\nEnglish: Betty drives fast .\nEnglish: Betty drives


100%|██████████| 3/3 [00:01<00:00,  1.80it/s]

[INFO] Done. Saved to dev10.gpt2.format.nl


This sometimes improves the output slightly. The model may follow the pattern for some sentences,
but behaviour remains inconsistent

# Step 3. Few-shot prompting 

We now provide examples in the prompt.

In [5]:
%run gpt2_translate.py \
  --input dev10.en \
  --output dev10.gpt2.fewshot.nl \
  --source-lang English \
  --target-lang Dutch \
  --model gpt2 \
  --prompt-template "English: I eat apples.\nDutch: Ik eet appels.\n\nEnglish: She is at home.\nDutch: Zij is thuis.\n\nEnglish: {text}\nDutch:" \
  --temperature 0 \
  --batch-size 4 \
  --strip-after-first-line \
  --show-translations \
  --show-n 5

[INFO] Loading model: gpt2


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Translating...


 33%|███▎      | 1/3 [00:00<00:01,  1.70it/s]


---
SRC: English: I eat apples.\nDutch: Ik eet appels.\n\nEnglish: She is at home.\nDutch: Zij is thuis.\n\nEnglish: Nobody reads about my country .\nDutch:
HYP: I am a Dutchman.\n\nEnglish: I am a Dutchman.\n\nEnglish: I am a Dutchman.\n\nEnglish: I am a Dutchman.\n\nEnglish: I am

---
SRC: English: I eat apples.\nDutch: Ik eet appels.\n\nEnglish: She is at home.\nDutch: Zij is thuis.\n\nEnglish: I sleep in my car .\nDutch:
HYP: I am a good cook.\n\nEnglish: I am a good cook.\n\nEnglish: I am a good cook.\n\nEnglish: I am a good cook.\n\nEnglish: I am

---
SRC: English: I eat apples.\nDutch: Ik eet appels.\n\nEnglish: She is at home.\nDutch: Zij is thuis.\n\nEnglish: There 're clean sheets under the bed .\nDutch:
HYP: Zij is at home.\n\nEnglish: There 're clean sheets under the bed .\nDutch: Zij is at home.\n\nEnglish: There 're clean sheets under the bed .\nDutch:

---
SRC: English: I eat apples.\nDutch: Ik eet appels.\n\nEnglish: She is at home.\nDutch: Zij is thuis.\n\nEnglish: I 

 67%|██████▋   | 2/3 [00:01<00:00,  1.72it/s]


---
SRC: English: I eat apples.\nDutch: Ik eet appels.\n\nEnglish: She is at home.\nDutch: Zij is thuis.\n\nEnglish: Betty drives fast .\nDutch:
HYP: Zij is thuis.\n\nEnglish: Betty is a good girl.\n\nEnglish: Betty is a good girl.\n\nEnglish: Betty is a good girl.\n\nEnglish: Betty


100%|██████████| 3/3 [00:01<00:00,  1.63it/s]

[INFO] Done. Saved to dev10.gpt2.fewshot.nl


Few-shot prompting makes the intended pattern more explicit, and the model is sometimes able
to produce partially correct translations.

However, the behaviour remains unstable:
* some outputs follow the pattern,
* others copy the input or break the structure,
* consistency across sentences is low

# Step 4. Stochastic decoding 

Finally, we explore the effect of sampling.

In [6]:
%run gpt2_translate.py \
  --input dev10.en \
  --output dev10.gpt2.sampled.nl \
  --source-lang English \
  --target-lang Dutch \
  --model gpt2 \
  --prompt-template "English: {text}\nDutch:" \
  --temperature 0.7 \
  --top-p 0.9 \
  --batch-size 4 \
  --strip-after-first-line \
  --show-translations \
  --show-n 5

[INFO] Loading model: gpt2


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Translating...


 33%|███▎      | 1/3 [00:00<00:01,  1.28it/s]


---
SRC: English: Nobody reads about my country .\nDutch:
HYP: I don't know about it, but I do know about it.\nEnglish: So, I just told you about it.\nEnglish: I will tell you about it, if you want.\nEnglish: I won't tell

---
SRC: English: I sleep in my car .\nDutch:
HYP: I will go home.\nEnglish: I'm sorry for having to say something to you.\nEnglish: I'm sorry for having to say something to you.\nEnglish: I'm sorry for having to say something to you.

---
SRC: English: There 're clean sheets under the bed .\nDutch:
HYP: I'm here to fix it up .\nEnglish: But if you want to fix it up, then you have to go to the bathroom, right?\nEnglish: And that's where the pipes are .\nEnglish: But then

---
SRC: English: I have a donkey .\nDutch:
HYP: I am a donkey.\nEtymology: From Dutch donkey.


 67%|██████▋   | 2/3 [00:01<00:00,  1.50it/s]


---
SRC: English: Betty drives fast .\nDutch:
HYP: Betty drives fast. \nEnglish: Betty drives fast.


100%|██████████| 3/3 [00:01<00:00,  1.57it/s]

[INFO] Done. Saved to dev10.gpt2.sampled.nl


Sampling introduces variation in the outputs. In some cases, this leads to more fluent fragments, but it often reduces adequacy and increases instability.

# Interpretation 

The behaviour observed in these experiments reflects the core properties of
decoder-only models.

GPT-2 does not perform translation as a controlled mapping from source to target. Instead, it
treats the prompt as a sequence to be continued. This leads to characteristic behaviours such as:
* repeating parts of the prompt,
* copying the source sentence,
* generating lists or unrelated continuations,
* producing fluent but incorrect text.

These observations illustrate that next-token prediction alone is not sufficient for reliable translation. Additional training stages, such as instruction tuning and alignment, are required to make
decoder-only models behave as usable translation systems.
